# Pipeline Données — Détection d'Émotions Multimodale

> **Projet M1** — Melissa / Asma / Kawther  
>
> Ce notebook couvre l'intégralité du pipeline données :
> 1. Téléchargement — RAVDESS (Zenodo) + CREMA-D (Kaggle / GitHub)
> 2. Parsing et fusion des labels vers 7 classes unifiées
> 3. Nettoyage — fichiers corrompus, durées aberrantes
> 4. Prétraitement audio — 16 kHz · mono · normalisation · VAD · padding · `attention_mask`
> 5. Split speaker-independent — 70 / 15 / 15
> 6. Sauvegarde — `.npy` + HuggingFace Arrow Dataset + CSV

---

## 0. Installation

In [31]:
!pip install torch torchaudio -q
!pip install transformers datasets huggingface-hub -q
!pip install librosa soundfile audiomentations -q
!pip install scikit-learn pandas numpy matplotlib seaborn -q
!pip install tqdm requests kaggle ipywidgets -q

## 1. Imports & Configuration

In [32]:
import os
import zipfile
import shutil
import requests
from pathlib import Path

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import torch
import torchaudio
import torchaudio.transforms as T

# ─── CHEMINS ─────────────────────────────────────────────────────────────────
BASE_DIR      = Path('.')
DATA_DIR      = BASE_DIR / 'data'
RAW_DIR       = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
SPLITS_DIR    = DATA_DIR / 'splits'

# ─── PARAMÈTRES AUDIO ────────────────────────────────────────────────────────
SAMPLE_RATE  = 16_000          # Hz cible
MAX_DURATION = 4.0             # secondes
MAX_LENGTH   = int(SAMPLE_RATE * MAX_DURATION)  # 64 000 samples

# ─── CLASSES UNIFIÉES (7 classes) ────────────────────────────────────────────
# calm de RAVDESS est fusionné dans neutral (acoustiquement proches)
EMOTION_LABELS = {
    'neutral'  : 0,
    'happy'    : 1,
    'sad'      : 2,
    'angry'    : 3,
    'fearful'  : 4,
    'disgust'  : 5,
    'surprised': 6,
}

RAVDESS_MAP = {
    '01': 'neutral',   # neutral
    '02': 'neutral',   # calm → fusionné dans neutral
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised',
}

CREMAD_MAP = {
    'NEU': 'neutral',
    'HAP': 'happy',
    'SAD': 'sad',
    'ANG': 'angry',
    'FEA': 'fearful',
    'DIS': 'disgust',
}

print(f'Config chargee  —  MAX_LENGTH = {MAX_LENGTH} samples ({MAX_DURATION}s @ {SAMPLE_RATE} Hz)')
print(f'Classes         —  {list(EMOTION_LABELS.keys())}')

Config chargee  —  MAX_LENGTH = 64000 samples (4.0s @ 16000 Hz)
Classes         —  ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']


/Users/macbookair/Downloads/APS_Emotion_AI/emotion_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
def create_directories():
    dirs = [
        RAW_DIR / 'RAVDESS',
        RAW_DIR / 'CREMA-D',
        PROCESSED_DIR / 'audio',
        PROCESSED_DIR / 'masks',
        PROCESSED_DIR / 'hf_dataset',
        SPLITS_DIR,
    ]
    for d in dirs:
        d.mkdir(parents=True, exist_ok=True)
    print('Structure des dossiers creee :')
    for d in dirs:
        print(f'  {d}')

create_directories()

Structure des dossiers creee :
  data/raw/RAVDESS
  data/raw/CREMA-D
  data/processed/audio
  data/processed/masks
  data/processed/hf_dataset
  data/splits


## 2. Téléchargement des Données

In [34]:
def download_file(url, dest_path, desc='Telechargement'):
    """Telecharge un fichier avec barre de progression."""
    dest_path = Path(dest_path)
    if dest_path.exists():
        print(f'  Deja present : {dest_path}')
        return dest_path
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    total = int(response.headers.get('content-length', 0))
    with open(dest_path, 'wb') as f, tqdm(
        desc=desc, total=total, unit='B', unit_scale=True, unit_divisor=1024
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            bar.update(len(chunk))
    return dest_path

In [ ]:
# ─── RAVDESS — Zenodo (Speech only, ~460 Mo) ─────────────────────────────────
import zipfile as _zf

RAVDESS_URL = 'https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip'
RAVDESS_ZIP = RAW_DIR / 'RAVDESS.zip'
RAVDESS_DIR = RAW_DIR / 'RAVDESS'

# Supprimer un zip corrompu si present
if RAVDESS_ZIP.exists():
    try:
        with _zf.ZipFile(RAVDESS_ZIP, 'r') as z:
            z.namelist()
    except Exception:
        print('Zip corrompu detecte, suppression...')
        RAVDESS_ZIP.unlink()

if len(list(RAVDESS_DIR.rglob('*.wav'))) > 100:
    print(f'RAVDESS deja present ({len(list(RAVDESS_DIR.rglob(chr(42)+".wav")))} fichiers)')
else:
    print('Telechargement RAVDESS (~460 Mo) depuis Zenodo...')
    download_file(RAVDESS_URL, RAVDESS_ZIP, desc='RAVDESS')

    try:
        with _zf.ZipFile(RAVDESS_ZIP, 'r') as z:
            z.namelist()
    except Exception:
        RAVDESS_ZIP.unlink()
        print('ERREUR : fichier telecharge invalide.')
        print('Telechargez manuellement : https://zenodo.org/records/1188976')
        print(f'Placez le zip dans : {RAVDESS_ZIP}')
        raise

    print('Extraction...')
    with _zf.ZipFile(RAVDESS_ZIP, 'r') as z:
        z.extractall(RAVDESS_DIR)
    RAVDESS_ZIP.unlink()
    n = len(list(RAVDESS_DIR.rglob('*.wav')))
    print(f'RAVDESS extrait : {n} fichiers')

Zip corrompu detecte, suppression...
Telechargement RAVDESS (~460 Mo) depuis Zenodo...


In [ ]:
# ─── CREMA-D — Kaggle (recommande) ou GitHub ─────────────────────────────────
# Option A (recommandee) : Kaggle API
#   1. Creez un compte sur https://www.kaggle.com
#   2. Allez dans Account > API > Create New Token  → telecharge kaggle.json
#   3. Placez-le dans ~/.kaggle/kaggle.json  (chmod 600)
#   4. Executez la cellule ci-dessous
#
# Option B : GitHub zip (~1.5 Go)
#   Decommentez le bloc GitHub ci-dessous

CREMAD_DIR = RAW_DIR / 'CREMA-D'

if len(list(CREMAD_DIR.rglob('*.wav'))) > 100:
    print(f'CREMA-D deja present ({len(list(CREMAD_DIR.rglob("*.wav")))} fichiers)')
else:
    # ── Option A : Kaggle ──────────────────────────────────────────────────────
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    if kaggle_json.exists():
        print('Telechargement CREMA-D via Kaggle...')
        try:
            import kaggle
            kaggle.api.authenticate()
            kaggle.api.dataset_download_files(
                'ejlok1/cremad',
                path=str(CREMAD_DIR),
                unzip=True,
                quiet=False
            )
            print('CREMA-D telecharge.')
        except Exception as e:
            print(f'Erreur Kaggle : {e}')
    else:
        # ── Option B : GitHub zip ──────────────────────────────────────────────
        print('kaggle.json introuvable — tentative via GitHub (~1.5 Go)...')
        GH_URL  = 'https://github.com/CheyneyComputerScience/CREMA-D/archive/refs/heads/master.zip'
        GH_ZIP  = RAW_DIR / 'CREMA-D.zip'
        try:
            download_file(GH_URL, GH_ZIP, desc='CREMA-D')
            print('Extraction...')
            with zipfile.ZipFile(GH_ZIP, 'r') as z:
                # Les .wav sont dans CREMA-D-master/AudioWAV/
                members = [m for m in z.namelist() if m.endswith('.wav')]
                for m in tqdm(members, desc='Extraction WAV'):
                    filename = Path(m).name
                    with z.open(m) as src, open(CREMAD_DIR / filename, 'wb') as dst:
                        shutil.copyfileobj(src, dst)
            GH_ZIP.unlink()
            print(f'CREMA-D extrait : {len(list(CREMAD_DIR.glob("*.wav")))} fichiers')
        except Exception as e:
            print(f'Erreur GitHub : {e}')
            print('\n─── TELECHARGEMENT MANUEL ────────────────────────────────────')
            print('Telechargez CREMA-D depuis : https://www.kaggle.com/datasets/ejlok1/cremad')
            print(f'Extrayez les .wav dans     : {CREMAD_DIR.resolve()}')
            print('Structure attendue         : data/raw/CREMA-D/*.wav')
            print('──────────────────────────────────────────────────────────────')

## 3. Parsing des Labels

In [ ]:
def parse_ravdess(ravdess_dir):
    """
    Extrait les metadonnees depuis les noms de fichiers RAVDESS.
    Format : 03-01-[emotion]-[intensite]-[phrase]-[rep]-[acteur].wav
    Emotions : 01=neutral, 02=calm, 03=happy, 04=sad,
               05=angry, 06=fearful, 07=disgust, 08=surprised
    """
    records = []
    for fp in Path(ravdess_dir).rglob('*.wav'):
        parts = fp.stem.split('-')
        if len(parts) < 7:
            continue
        emotion_code = parts[2]
        actor_id     = int(parts[6])
        emotion_name = RAVDESS_MAP.get(emotion_code)
        if emotion_name is None:
            continue
        records.append({
            'filepath' : str(fp.resolve()),
            'emotion'  : emotion_name,
            'label'    : EMOTION_LABELS[emotion_name],
            'actor_id' : f'R_{actor_id:03d}',
            'dataset'  : 'RAVDESS',
        })
    df = pd.DataFrame(records)
    print(f'RAVDESS : {len(df)} fichiers | {df["emotion"].nunique()} emotions')
    print(df['emotion'].value_counts().to_string())
    return df

df_ravdess = parse_ravdess(RAW_DIR / 'RAVDESS')
df_ravdess.head(3)

In [ ]:
def parse_cremad(cremad_dir):
    """
    Extrait les metadonnees depuis les noms de fichiers CREMA-D.
    Format : [acteur]_[phrase]_[emotion]_[niveau].wav
    Emotions : NEU, HAP, SAD, ANG, FEA, DIS
    """
    records = []
    for fp in Path(cremad_dir).rglob('*.wav'):
        parts = fp.stem.split('_')
        if len(parts) < 3:
            continue
        actor_id     = parts[0]
        emotion_code = parts[2].upper()
        emotion_name = CREMAD_MAP.get(emotion_code)
        if emotion_name is None:
            continue
        records.append({
            'filepath' : str(fp.resolve()),
            'emotion'  : emotion_name,
            'label'    : EMOTION_LABELS[emotion_name],
            'actor_id' : f'C_{actor_id}',
            'dataset'  : 'CREMA-D',
        })
    df = pd.DataFrame(records)
    print(f'CREMA-D : {len(df)} fichiers | {df["emotion"].nunique()} emotions')
    print(df['emotion'].value_counts().to_string())
    return df

df_cremad = parse_cremad(RAW_DIR / 'CREMA-D')
df_cremad.head(3)

## 4. Fusion & Mapping des Classes

In [ ]:
df_combined = pd.concat([df_ravdess, df_cremad], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset fusionne : {len(df_combined)} fichiers')
print(f'Classes          : {sorted(df_combined["emotion"].unique())}')
print(f'Acteurs uniques  : {df_combined["actor_id"].nunique()}')
print(f'\nRepartition par classe :')
print(df_combined['emotion'].value_counts().to_string())
df_combined.head(5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Distribution globale ──────────────────────────────────────────────────────
counts = df_combined['emotion'].value_counts().sort_values()
colors = sns.color_palette('husl', len(counts))
bars = axes[0].barh(counts.index, counts.values, color=colors)
axes[0].set_title('Distribution des classes (fusionne)', fontsize=13)
axes[0].set_xlabel('Nombre de fichiers')
for bar, val in zip(bars, counts.values):
    axes[0].text(val + 30, bar.get_y() + bar.get_height()/2,
                 str(val), va='center', fontsize=9)

# ── Par dataset ───────────────────────────────────────────────────────────────
pivot = df_combined.groupby(['emotion', 'dataset']).size().unstack(fill_value=0)
pivot.plot(kind='bar', ax=axes[1], colormap='Set2', rot=30)
axes[1].set_title('Repartition par dataset et emotion', fontsize=13)
axes[1].set_xlabel('')
axes[1].legend(title='Dataset')

plt.tight_layout()
plt.savefig(DATA_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegarde dans data/class_distribution.png')

## 5. Nettoyage

In [ ]:
def check_audio(filepath):
    """Verifie qu'un fichier est lisible et retourne sa duree en secondes."""
    try:
        info = torchaudio.info(filepath)
        duration = info.num_frames / info.sample_rate
        return True, round(duration, 3)
    except Exception:
        return False, 0.0

print(f'Verification de {len(df_combined)} fichiers...')
valid_flags = []
durations   = []

for fp in tqdm(df_combined['filepath'], desc='Verification'):
    ok, dur = check_audio(fp)
    valid_flags.append(ok)
    durations.append(dur)

df_combined['valid']    = valid_flags
df_combined['duration'] = durations

n_invalid = (~df_combined['valid']).sum()
print(f'Fichiers invalides / corrompus : {n_invalid}')
if n_invalid > 0:
    print(df_combined[~df_combined['valid']][['filepath','dataset']].to_string())

# Supprimer les invalides
df_combined = df_combined[df_combined['valid']].copy()
df_combined.drop(columns=['valid'], inplace=True)

# Statistiques de duree
print(f'\nStatistiques de duree (secondes) par dataset :')
print(df_combined.groupby('dataset')['duration'].describe().round(2).to_string())

# Filtrer les fichiers < 0.5s (trop courts pour etre utiles)
too_short = (df_combined['duration'] < 0.5).sum()
df_combined = df_combined[df_combined['duration'] >= 0.5].copy()
print(f'\nFichiers < 0.5s supprimes : {too_short}')
print(f'Dataset apres nettoyage   : {len(df_combined)} fichiers')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for dataset, grp in df_combined.groupby('dataset'):
    ax.hist(grp['duration'], bins=40, alpha=0.6, label=dataset)
ax.axvline(MAX_DURATION, color='red', linestyle='--', label=f'max_length = {MAX_DURATION}s')
ax.set_title('Distribution des durees audio')
ax.set_xlabel('Duree (secondes)')
ax.set_ylabel('Nombre de fichiers')
ax.legend()
plt.tight_layout()
plt.savefig(DATA_DIR / 'duration_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Prétraitement Audio

Pipeline appliqué à chaque fichier :

| Étape | Outil | Détail |
|-------|-------|--------|
| Rééchantillonnage | `torchaudio.transforms.Resample` | → 16 000 Hz |
| Mono | `mean(dim=0)` | Moyenne des canaux |
| Normalisation | division par `max(abs)` | Signal dans [-1, 1] |
| VAD / trim silence | `librosa.effects.trim` | top_db = 30 |
| Padding / Troncature | `F.pad` / slicing | → 64 000 samples |
| Attention mask | vecteur binaire 0/1 | 1 = signal réel, 0 = padding |

In [ ]:
import torch.nn.functional as F


def load_and_resample(filepath, target_sr=SAMPLE_RATE):
    waveform, sr = torchaudio.load(filepath)
    if sr != target_sr:
        resampler = T.Resample(orig_freq=sr, new_freq=target_sr)
        waveform  = resampler(waveform)
    return waveform  # shape (channels, samples)


def to_mono(waveform):
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    return waveform  # shape (1, samples)


def normalize_amplitude(waveform):
    max_val = waveform.abs().max()
    if max_val > 1e-6:
        waveform = waveform / max_val
    return waveform


def trim_silence(waveform, top_db=30):
    """Supprime les silences en debut et fin (VAD energetique)."""
    audio_np = waveform.squeeze().numpy()
    trimmed, _ = librosa.effects.trim(audio_np, top_db=top_db)
    if len(trimmed) < 160:          # garder au moins 10ms
        trimmed = audio_np
    return torch.tensor(trimmed, dtype=torch.float32).unsqueeze(0)


def pad_or_truncate(waveform, max_length=MAX_LENGTH):
    """
    Ajuste la longueur a max_length.
    Retourne (waveform_padded, attention_mask).

    attention_mask :
        1 = position contient du vrai signal
        0 = position contient du padding (zeros)
    """
    length = waveform.shape[-1]

    if length >= max_length:
        waveform_out = waveform[..., :max_length]
        mask = torch.ones(max_length, dtype=torch.uint8)
    else:
        pad_size     = max_length - length
        waveform_out = F.pad(waveform, (0, pad_size))
        mask = torch.cat([
            torch.ones(length,   dtype=torch.uint8),
            torch.zeros(pad_size, dtype=torch.uint8)
        ])

    return waveform_out, mask


def preprocess_audio(filepath):
    """
    Pipeline complet pour un fichier audio.

    Retourne :
        waveform  : np.ndarray float32 — shape (MAX_LENGTH,)
        mask      : np.ndarray uint8   — shape (MAX_LENGTH,)
        real_len  : int — samples de vrai signal avant padding
    """
    waveform = load_and_resample(filepath)
    waveform = to_mono(waveform)
    waveform = normalize_amplitude(waveform)
    waveform = trim_silence(waveform)
    waveform, mask = pad_or_truncate(waveform)

    real_len = int(mask.sum().item())
    return (
        waveform.squeeze().numpy().astype(np.float32),
        mask.numpy().astype(np.uint8),
        real_len,
    )


print('Fonctions de pretraitement definies.')
print(f'Sortie attendue : waveform ({MAX_LENGTH},) float32  +  mask ({MAX_LENGTH},) uint8')

In [ ]:
AUDIO_OUT = PROCESSED_DIR / 'audio'
MASK_OUT  = PROCESSED_DIR / 'masks'
AUDIO_OUT.mkdir(parents=True, exist_ok=True)
MASK_OUT.mkdir(parents=True, exist_ok=True)

processed_paths = []
mask_paths      = []
real_lengths    = []
errors          = []

print(f'Pretraitement de {len(df_combined)} fichiers...')
print(f'  Waveforms -> {AUDIO_OUT}')
print(f'  Masks     -> {MASK_OUT}\n')

for i, (idx, row) in enumerate(tqdm(
    df_combined.iterrows(), total=len(df_combined), desc='Preprocessing'
)):
    name     = f'{row["dataset"]}_{row["actor_id"]}_{i:05d}.npy'
    wav_out  = AUDIO_OUT / name
    mask_out = MASK_OUT  / name

    try:
        wave, mask, real_len = preprocess_audio(row['filepath'])
        np.save(wav_out,  wave)
        np.save(mask_out, mask)
        processed_paths.append(str(wav_out))
        mask_paths.append(str(mask_out))
        real_lengths.append(real_len)
    except Exception as e:
        errors.append((idx, str(e)))
        processed_paths.append('')
        mask_paths.append('')
        real_lengths.append(0)

df_combined['processed_path'] = processed_paths
df_combined['mask_path']      = mask_paths
df_combined['real_length']    = real_lengths

# Supprimer les lignes en erreur
df_combined = df_combined[df_combined['real_length'] > 0].copy()

if errors:
    print(f'\nErreurs de pretraitement : {len(errors)}')
else:
    print(f'\nTous les fichiers traites sans erreur.')
print(f'Dataset final : {len(df_combined)} fichiers')

In [ ]:
# ─── Verification visuelle sur 3 echantillons ─────────────────────────────────
sample = df_combined.sample(3, random_state=7)
fig, axes = plt.subplots(3, 2, figsize=(14, 9))

for i, (_, row) in enumerate(sample.iterrows()):
    wave = np.load(row['processed_path'])
    mask = np.load(row['mask_path'])
    t    = np.arange(MAX_LENGTH) / SAMPLE_RATE
    real_t = row['real_length'] / SAMPLE_RATE

    # Waveform
    axes[i, 0].plot(t, wave, linewidth=0.4, color='steelblue')
    axes[i, 0].axvline(real_t, color='red', linestyle='--',
                       linewidth=1.2, label=f'fin signal ({real_t:.2f}s)')
    axes[i, 0].fill_between(t, wave, alpha=0.2, color='steelblue')
    axes[i, 0].set_title(f'{row["emotion"]} | {row["dataset"]} | {row["actor_id"]}')
    axes[i, 0].set_xlabel('Temps (s)')
    axes[i, 0].set_ylabel('Amplitude')
    axes[i, 0].legend(fontsize=8)

    # Attention mask
    axes[i, 1].fill_between(t, mask, step='post', alpha=0.7, color='orange')
    axes[i, 1].set_title(f'attention_mask  (real={row["real_length"]} / {MAX_LENGTH})')
    axes[i, 1].set_xlabel('Temps (s)')
    axes[i, 1].set_ylim(-0.05, 1.15)

plt.tight_layout()
plt.savefig(DATA_DIR / 'waveform_samples.png', dpi=150, bbox_inches='tight')
plt.show()

wave = np.load(sample.iloc[0]['processed_path'])
mask = np.load(sample.iloc[0]['mask_path'])
print(f'Shape waveform : {wave.shape}  dtype={wave.dtype}')
print(f'Shape mask     : {mask.shape}  dtype={mask.dtype}')
print(f'Valeurs mask   : {set(mask.tolist())}')

## 7. Split Speaker-Independent

Aucun acteur ne se retrouve à la fois dans le train et dans le test/val.
Split : **70 % train — 15 % val — 15 % test**.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

print(f'Acteurs uniques : {df_combined["actor_id"].nunique()}')
print(f'Total fichiers  : {len(df_combined)}')

# Etape 1 — train vs (val + test)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(
    df_combined,
    df_combined['label'],
    groups=df_combined['actor_id']
))
df_train = df_combined.iloc[train_idx].copy()
df_temp  = df_combined.iloc[temp_idx].copy()

# Etape 2 — val vs test (50/50 sur temp → 15% chacun)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(
    df_temp,
    df_temp['label'],
    groups=df_temp['actor_id']
))
df_val  = df_temp.iloc[val_idx].copy()
df_test = df_temp.iloc[test_idx].copy()

# ─── Validation : aucun acteur en commun ──────────────────────────────────────
assert not set(df_train['actor_id']) & set(df_test['actor_id']), 'Fuite train/test !'
assert not set(df_val['actor_id'])   & set(df_test['actor_id']), 'Fuite val/test !'
assert not set(df_train['actor_id']) & set(df_val['actor_id']),  'Fuite train/val !'
print('Speaker-independent valide — aucun acteur en commun entre splits.')

# ─── Sauvegarder les splits ───────────────────────────────────────────────────
df_train.to_csv(SPLITS_DIR / 'train.csv', index=False)
df_val.to_csv(  SPLITS_DIR / 'val.csv',   index=False)
df_test.to_csv( SPLITS_DIR / 'test.csv',  index=False)

print(f'\nRepartition :')
print(f'  Train : {len(df_train):5d} fichiers  ({df_train["actor_id"].nunique()} acteurs)')
print(f'  Val   : {len(df_val):5d} fichiers  ({df_val["actor_id"].nunique()} acteurs)')
print(f'  Test  : {len(df_test):5d} fichiers  ({df_test["actor_id"].nunique()} acteurs)')

# ─── Distribution des classes par split ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, df_s) in zip(axes, [('Train', df_train), ('Val', df_val), ('Test', df_test)]):
    counts = df_s['emotion'].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=sns.color_palette('husl', len(counts)))
    ax.set_title(f'{name} ({len(df_s)} fichiers)')
    ax.set_xlabel('Emotion')
    ax.tick_params(axis='x', rotation=30)
    if ax == axes[0]:
        ax.set_ylabel('Nombre de fichiers')
plt.tight_layout()
plt.savefig(DATA_DIR / 'split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Sauvegarde — HuggingFace Arrow Dataset

Le dataset Arrow permet un rechargement instantané sans recalculer le prétraitement.

In [ ]:
from datasets import Dataset, DatasetDict

def df_to_hf(df, split_name=''):
    """Convertit un DataFrame en HuggingFace Dataset."""
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'HF {split_name}'):
        waveform = np.load(row['processed_path'])
        mask     = np.load(row['mask_path'])
        records.append({
            'input_values'   : waveform.tolist(),
            'attention_mask' : mask.tolist(),
            'label'          : int(row['label']),
            'emotion'        : str(row['emotion']),
            'actor_id'       : str(row['actor_id']),
            'dataset'        : str(row['dataset']),
            'real_length'    : int(row['real_length']),
        })
    return Dataset.from_list(records)

print('Creation du HuggingFace DatasetDict...')
hf_dataset = DatasetDict({
    'train': df_to_hf(df_train, 'train'),
    'val'  : df_to_hf(df_val,   'val'),
    'test' : df_to_hf(df_test,  'test'),
})

print(hf_dataset)

hf_path = str(PROCESSED_DIR / 'hf_dataset')
hf_dataset.save_to_disk(hf_path)
print(f'\nDataset sauvegarde dans : {hf_path}')
print('Rechargement ulterieur  :')
print('  from datasets import load_from_disk')
print(f'  dataset = load_from_disk("{hf_path}")')

## 9. Résumé Final

In [ ]:
total = len(df_train) + len(df_val) + len(df_test)

print('=' * 62)
print('   PIPELINE DONNEES — TERMINE')
print('=' * 62)
print(f'  Total fichiers traites  : {total}')
print(f'  Train / Val / Test      : {len(df_train)} / {len(df_val)} / {len(df_test)}')
print(f'  Frequence echantillon   : {SAMPLE_RATE} Hz')
print(f'  Duree fixe              : {MAX_DURATION}s ({MAX_LENGTH} samples)')
print(f'  Nombre de classes       : {len(EMOTION_LABELS)}')
print(f'  Classes                 : {list(EMOTION_LABELS.keys())}')
print()
print('  Fichiers produits :')
print(f'  {"data/processed/audio/":<40} waveforms .npy (float32)')
print(f'  {"data/processed/masks/":<40} attention masks .npy (uint8)')
print(f'  {"data/processed/hf_dataset/":<40} HuggingFace Arrow Dataset')
print(f'  {"data/splits/train.csv":<40} split train')
print(f'  {"data/splits/val.csv":<40} split validation')
print(f'  {"data/splits/test.csv":<40} split test')
print()
print('  Distribution par classe et split :')
for split_name, df_s in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    counts = df_s['emotion'].value_counts().sort_index()
    print(f'\n  [{split_name}]')
    for emotion, count in counts.items():
        bar = chr(9608) * (count // 60)
        print(f'    {emotion:<12} {count:5d}  {bar}')
print()
print('  Pipeline termine. Donnees pretes pour l\'entrainement.')
print('=' * 62)